# Notebook 2 - Chess: Implementing Pieces, Board, Move and Game

In Notebook 1 we designed the classes. Now let's **build** them and watch the design pay off.

**Plan**

1. `Piece` hierarchy - King, Queen, Rook, Bishop, Knight, Pawn.
2. `Board` - a thin 8x8 container with pretty-printing.
3. `Move` - a tiny data class so we can **undo** moves.
4. `Game` - turn order, legal-move checks, **basic check detection**, move history.
5. A runnable mini-game plus common real-world rules (pawn's first double step, captures).

> Everything runs. Each cell builds on the previous one - run them in order.


## Setup

```bash
cd 07-object-oriented-design/chess
uv sync
```

In VS Code pick the `.venv` kernel (top-right). If it doesn't appear, reload the window
(`Cmd+Shift+P` then **Reload Window**).


## 1. The `Piece` hierarchy

All six pieces in one cell so it's easy to scan. Note how **each class owns its own rules**.
Also notice the shared `slide` helper - Rook, Bishop, and Queen are all "sliding pieces",
so we write the ray-casting logic **once**.


In [ ]:
from abc import ABC, abstractmethod

WHITE, BLACK = 'W', 'B'

def on_board(r, c):
    return 0 <= r < 8 and 0 <= c < 8


class Piece(ABC):
    def __init__(self, color):
        self.color = color

    @abstractmethod
    def symbol(self) -> str: ...

    @abstractmethod
    def valid_moves(self, board, r, c): ...

    def __repr__(self):
        s = self.symbol()
        return s.upper() if self.color == WHITE else s.lower()


# ---- Helper for sliding pieces (Rook, Bishop, Queen) ----
def slide(board, r, c, color, directions):
    out = []
    for dr, dc in directions:
        nr, nc = r+dr, c+dc
        while on_board(nr, nc):
            if board[nr][nc] is None:
                out.append((nr, nc))
            else:
                if board[nr][nc].color != color:
                    out.append((nr, nc))  # capture
                break
            nr += dr; nc += dc
    return out


class King(Piece):
    def symbol(self): return 'k'
    def valid_moves(self, board, r, c):
        out = []
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if dr == 0 and dc == 0:
                    continue
                nr, nc = r+dr, c+dc
                if on_board(nr, nc) and (board[nr][nc] is None or board[nr][nc].color != self.color):
                    out.append((nr, nc))
        return out


class Rook(Piece):
    def symbol(self): return 'r'
    def valid_moves(self, board, r, c):
        return slide(board, r, c, self.color, [(-1,0),(1,0),(0,-1),(0,1)])


class Bishop(Piece):
    def symbol(self): return 'b'
    def valid_moves(self, board, r, c):
        return slide(board, r, c, self.color, [(-1,-1),(-1,1),(1,-1),(1,1)])


class Queen(Piece):
    """Queen = Rook + Bishop. Literally."""
    def symbol(self): return 'q'
    def valid_moves(self, board, r, c):
        return slide(board, r, c, self.color,
                     [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)])


class Knight(Piece):
    def symbol(self): return 'n'
    def valid_moves(self, board, r, c):
        deltas = [(-2,-1),(-2,1),(-1,-2),(-1,2),(1,-2),(1,2),(2,-1),(2,1)]
        out = []
        for dr, dc in deltas:
            nr, nc = r+dr, c+dc
            if on_board(nr, nc) and (board[nr][nc] is None or board[nr][nc].color != self.color):
                out.append((nr, nc))
        return out


class Pawn(Piece):
    """Pawns are weird: forward 1, forward 2 on first move, capture diagonally."""
    def symbol(self): return 'p'
    def valid_moves(self, board, r, c):
        dr = -1 if self.color == WHITE else 1   # white moves toward row 0
        start_row = 6 if self.color == WHITE else 1
        out = []
        # forward 1
        if on_board(r+dr, c) and board[r+dr][c] is None:
            out.append((r+dr, c))
            # forward 2 from starting row
            if r == start_row and board[r+2*dr][c] is None:
                out.append((r+2*dr, c))
        # diagonal captures
        for dc in (-1, 1):
            nr, nc = r+dr, c+dc
            if on_board(nr, nc) and board[nr][nc] and board[nr][nc].color != self.color:
                out.append((nr, nc))
        return out

print('Pieces defined:', [c.__name__ for c in (King, Queen, Rook, Bishop, Knight, Pawn)])


## 2. The `Board` - a thin container

The board does **not** know chess rules. It only:

- holds an 8x8 grid,
- sets up the starting position,
- prints itself.

That's the Single Responsibility Principle: one reason to change.

> **Why do pieces receive `board.grid` and not the `Board` itself?**
> Dependency direction. `Board` knows about pieces (it stores them); if `Piece`
> also imported `Board`, the two would be locked together and you could not test
> a rook without constructing a whole `Board`. Handing pieces the plain 8×8 grid
> — the smallest thing they actually need — keeps the arrow pointing one way:
> `Game → Board → Piece`, never back. This is the Interface Segregation idea in
> miniature: depend on the least you can get away with.


In [ ]:
class Board:
    FILES = 'abcdefgh'

    def __init__(self):
        self.grid = [[None]*8 for _ in range(8)]

    def setup_start(self):
        back = [Rook, Knight, Bishop, Queen, King, Bishop, Knight, Rook]
        for c, cls in enumerate(back):
            self.grid[0][c] = cls(BLACK)
            self.grid[7][c] = cls(WHITE)
        for c in range(8):
            self.grid[1][c] = Pawn(BLACK)
            self.grid[6][c] = Pawn(WHITE)
        return self

    def __getitem__(self, rc):
        r, c = rc
        return self.grid[r][c]

    def __setitem__(self, rc, value):
        r, c = rc
        self.grid[r][c] = value

    def show(self):
        print('   ' + ' '.join(self.FILES))
        for r, row in enumerate(self.grid):
            rank = 8 - r
            line = ' '.join(str(p) if p else '.' for p in row)
            print(f'{rank}  {line}  {rank}')
        print('   ' + ' '.join(self.FILES))

b = Board().setup_start()
b.show()


## 3. The `Move` data class - so we can undo

A `Move` is just a record: *from where, to where, and what (if anything) was captured*.
Holding onto the captured piece lets us **undo** - which is exactly how chess engines
explore "what if I play this?" without actually committing.


In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class Move:
    frm: tuple            # (row, col)
    to: tuple             # (row, col)
    piece: Piece          # what moved
    captured: Optional[Piece] = None   # what was taken, if anything

    def __repr__(self):
        files = 'abcdefgh'
        def sq(rc):
            r, c = rc
            return f'{files[c]}{8 - r}'
        cap = 'x' if self.captured else '-'
        return f'{self.piece}{sq(self.frm)}{cap}{sq(self.to)}'

m = Move(frm=(6,4), to=(4,4), piece=Pawn(WHITE))
print('Example move:', m)


## 4. The `Game` - turn order, making moves, undo, check detection

Now we combine everything. The `Game`:

- keeps whose turn it is,
- validates a move by asking the piece,
- records moves in history,
- supports **undo**,
- detects a simple **"is the king in check?"** condition.

> **Check detection intuition:** the king is "in check" if *any* enemy piece's valid moves
> include the king's square. We already have `valid_moves` on every piece - reuse it!


In [ ]:
class Game:
    def __init__(self, board=None):
        self.board = board or Board().setup_start()
        self.turn = WHITE
        self.history: list = []

    # ----- locating the king (needed for check detection) -----
    def _find_king(self, color):
        for r in range(8):
            for c in range(8):
                p = self.board[r, c]
                if isinstance(p, King) and p.color == color:
                    return (r, c)
        return None

    def in_check(self, color):
        king_pos = self._find_king(color)
        if king_pos is None:
            return False
        enemy = BLACK if color == WHITE else WHITE
        for r in range(8):
            for c in range(8):
                p = self.board[r, c]
                if p and p.color == enemy:
                    if king_pos in p.valid_moves(self.board.grid, r, c):
                        return True
        return False

    # ----- making and undoing moves -----
    def move(self, frm, to):
        r, c = frm; nr, nc = to
        p = self.board[r, c]
        if p is None:
            raise ValueError(f'no piece at {frm}')
        if p.color != self.turn:
            raise ValueError(f"it is {self.turn}'s turn, not {p.color}'s")
        if (nr, nc) not in p.valid_moves(self.board.grid, r, c):
            raise ValueError(f'illegal move for {p}: {frm} -> {to}')

        captured = self.board[nr, nc]
        self.board[nr, nc] = p
        self.board[r, c] = None

        # Rule: a move cannot leave YOUR OWN king in check.
        if self.in_check(self.turn):
            self.board[r, c] = p
            self.board[nr, nc] = captured
            raise ValueError('illegal: would leave your king in check')

        self.history.append(Move(frm, to, p, captured))
        self.turn = BLACK if self.turn == WHITE else WHITE

    def undo(self):
        if not self.history:
            raise ValueError('nothing to undo')
        m = self.history.pop()
        r, c = m.frm; nr, nc = m.to
        self.board[r, c] = m.piece
        self.board[nr, nc] = m.captured
        self.turn = m.piece.color   # back to the mover's turn

print('Game class ready.')


## 5. Play a tiny opening

Let's play a few moves, print the board, then check and undo.

> **Coordinate reminder:** our grid uses `(row, col)` with row 0 at the top (Black's side).
> So White's `e2` pawn is at `(6, 4)` and moving to `e4` means `(4, 4)`.


In [ ]:
g = Game()
print('Starting position:')
g.board.show()

# A famous opening: 1. e4 e5  2. Nf3 Nc6
g.move((6,4), (4,4))   # white:  e2 -> e4
g.move((1,4), (3,4))   # black:  e7 -> e5
g.move((7,6), (5,5))   # white:  Ng1 -> f3
g.move((0,1), (2,2))   # black:  Nb8 -> c6

print()
print('After 4 moves:')
g.board.show()
print('Move history:')
for m in g.history:
    print(' ', m)
print('White in check?', g.in_check(WHITE))


### Undo

In [ ]:
g.undo()
g.undo()
print('After undoing 2 moves:')
g.board.show()
print('Moves remaining in history:', len(g.history))
print('Whose turn:', g.turn)


### Illegal moves raise clear errors

Great error messages are part of good OOD: the user (or tests) should know *why* a move failed.


In [ ]:
def try_move(g, frm, to, label):
    try:
        g.move(frm, to)
        print(f'{label}: OK')
    except ValueError as e:
        print(f'{label}: rejected -> {e}')

try_move(g, (7,1), (6,0), 'knight to non-L square')   # illegal shape
try_move(g, (1,0), (2,0), 'moving opponent piece')    # wrong color
try_move(g, (6,4), (3,4), 'pawn jumping too far')     # pawn cannot leap 3


## 6. A tiny "in check" demo

Let's construct a position where Black's king is attacked by a White rook,
and confirm `in_check` returns `True`.


In [ ]:
demo = Board()
demo.grid[7][4] = King(WHITE)   # white king on e1
demo.grid[0][4] = King(BLACK)   # black king on e8
demo.grid[4][4] = Rook(WHITE)   # rook on e4, same file as black king
demo.show()

g2 = Game(demo)
g2.turn = BLACK   # it's black's turn to respond to check
print('Black in check?', g2.in_check(BLACK))
print('White in check?', g2.in_check(WHITE))


## 7. Verify the design

Every claim we made — "each piece owns its rules", "`Move` makes undo trivial",
"`in_check` reuses `valid_moves`" — is testable. Below, each block checks one of
them. Notice how *easy* the tests are to write: that ease is the payoff of the
design. Try writing the same tests against `BadChess` from Notebook 1.

In [ ]:
def empty():
    return [[None]*8 for _ in range(8)]

def moves(piece, r, c, grid=None):
    grid = grid if grid is not None else empty()
    grid[r][c] = piece
    return set(piece.valid_moves(grid, r, c))

# ── Each piece owns its own geometry, and it is correct ─────────────────
# On an empty board a rook always sees 14 squares, a bishop 7 from a corner
# and 13 from the centre, a queen the sum of the two.
assert len(moves(Rook(WHITE),   7, 0)) == 14
assert len(moves(Rook(WHITE),   4, 4)) == 14
assert len(moves(Bishop(WHITE), 7, 0)) == 7
assert len(moves(Bishop(WHITE), 4, 4)) == 13
assert len(moves(Queen(WHITE),  4, 4)) == 27          # 14 + 13
assert len(moves(Knight(WHITE), 0, 0)) == 2           # corner
assert len(moves(Knight(WHITE), 4, 4)) == 8           # centre
assert len(moves(King(WHITE),   0, 0)) == 3
assert len(moves(King(WHITE),   4, 4)) == 8
assert (4, 4) not in moves(King(WHITE), 4, 4), "a piece may not 'move' to its own square"
assert all(on_board(r, c) for r, c in moves(Queen(WHITE), 0, 0)), "no move may leave the board"

# ── Sliders stop at the first obstacle; they never jump ─────────────────
g = empty()
g[4][6] = Pawn(BLACK)          # an enemy on the same rank
g[4][2] = Pawn(WHITE)          # a friend on the same rank
rook_moves = moves(Rook(WHITE), 4, 4, g)
assert (4, 6) in rook_moves,     "an enemy square is a capture"
assert (4, 7) not in rook_moves, "a slider must not jump over a piece it captures"
assert (4, 2) not in rook_moves, "a friendly piece is not capturable"
assert (4, 3) in rook_moves,     "…but the empty square in front of it is reachable"
assert (4, 1) not in rook_moves

# ── Pawns are the special case, so they get the most checks ─────────────
assert moves(Pawn(WHITE), 6, 4) == {(5, 4), (4, 4)}   # 1 or 2 from the start rank
assert moves(Pawn(WHITE), 5, 4) == {(4, 4)}           # only 1 once it has moved
assert moves(Pawn(BLACK), 1, 4) == {(2, 4), (3, 4)}   # black moves the other way
g = empty(); g[5][4] = Pawn(BLACK)                    # blocked head-on
assert moves(Pawn(WHITE), 6, 4, g) == set(),          "a pawn cannot capture forwards"
g = empty(); g[4][4] = Pawn(BLACK)                    # blocker two squares ahead
assert moves(Pawn(WHITE), 6, 4, g) == {(5, 4)},       "the double step needs a clear path"
g = empty(); g[5][3] = Pawn(BLACK); g[5][5] = Pawn(WHITE)
assert moves(Pawn(WHITE), 6, 4, g) == {(5, 4), (4, 4), (5, 3)}, \
    "captures diagonally, and only enemies"

# ── The Board sets itself up correctly ─────────────────────────────────
b = Board().setup_start()
assert sum(1 for r in range(8) for c in range(8) if b[r, c]) == 32
assert isinstance(b[7, 4], King) and b[7, 4].color == WHITE   # white king on e1
assert isinstance(b[0, 3], Queen) and b[0, 3].color == BLACK  # black queen on d8
assert all(isinstance(b[6, c], Pawn) for c in range(8))
assert all(b[r, c] is None for r in (2, 3, 4, 5) for c in range(8))

# ── Game enforces turn order and rejects illegal moves ─────────────────
def rejects(g, frm, to, why):
    try:
        g.move(frm, to)
    except ValueError:
        return True
    raise AssertionError(f"should have been rejected: {why}")

g = Game()
rejects(g, (1, 4), (2, 4), "black moving first")
rejects(g, (5, 4), (4, 4), "no piece on that square")
rejects(g, (7, 0), (5, 0), "rook blocked by its own pawn")
rejects(g, (6, 4), (3, 4), "pawn leaping three squares")
g.move((6, 4), (4, 4))
assert g.turn == BLACK, "the turn must pass after a legal move"
rejects(g, (4, 4), (3, 4), "white moving twice in a row")

# ── Move + undo restore the position EXACTLY (captures included) ───────
g = Game()
snapshot = [[g.board[r, c] for c in range(8)] for r in range(8)]
g.move((6, 4), (4, 4)); g.move((1, 3), (3, 3))    # 1. e4 d5
g.move((4, 4), (3, 3))                            # 2. exd5 — a capture
captured = g.history[-1].captured
assert isinstance(captured, Pawn) and captured.color == BLACK
assert len(g.history) == 3
for _ in range(3):
    g.undo()
assert [[g.board[r, c] for c in range(8)] for r in range(8)] == snapshot, \
    "undo must put every piece back, including the captured one"
assert g.turn == WHITE and g.history == []
try:
    g.undo(); raise AssertionError("undo on an empty history should raise")
except ValueError:
    pass

# ── Check detection, and the rule that follows from it ────────────────
pos = Board()
pos.grid[7][4] = King(WHITE)      # e1
pos.grid[0][4] = King(BLACK)      # e8
pos.grid[4][4] = Rook(WHITE)      # e4: attacks straight up the e-file
g = Game(pos)
assert g.in_check(BLACK) and not g.in_check(WHITE)

# A king may not walk into check — King.valid_moves offers the square,
# Game.move refuses it. Two layers: geometry vs legality.
pos = Board()
pos.grid[7][4] = King(WHITE)
pos.grid[0][3] = Rook(BLACK)      # owns the d-file
g = Game(pos)
assert (7, 3) in pos[7, 4].valid_moves(pos.grid, 7, 4), "geometry allows it"
rejects(g, (7, 4), (7, 3), "king stepping onto an attacked square")
assert pos[7, 4] is not None and pos[7, 3] is None, "a rejected move must not move anything"

# A pinned piece is frozen even though its own geometry says otherwise.
pos = Board()
pos.grid[7][4] = King(WHITE)      # e1
pos.grid[6][4] = Rook(WHITE)      # e2 — the only thing shielding the king
pos.grid[0][4] = Rook(BLACK)      # e8
g = Game(pos)
assert not g.in_check(WHITE), "not in check while the shield stands"
assert (6, 0) in pos[6, 4].valid_moves(pos.grid, 6, 4)
rejects(g, (6, 4), (6, 0), "moving a pinned rook off the pin line")
g.move((6, 4), (5, 4))            # sliding UP the pin line is fine
assert not g.in_check(WHITE)

# ── Move renders as readable notation ─────────────────────────────────
assert repr(Move((6, 4), (4, 4), Pawn(WHITE))) == "Pe2-e4"
assert repr(Move((4, 4), (3, 3), Pawn(WHITE), Pawn(BLACK))) == "Pe4xd5"

print("✅ piece geometry, board setup, turn order, undo, check and pins all verified")

## 7. What we still skipped (and why)

These are great **exercises** for you to extend:

- **Castling** - needs "have I moved yet?" flags on King/Rook and path checks.
- **En-passant** - needs to know the *previous* move (we already have `history`!).
- **Pawn promotion** - when a pawn reaches the last rank, swap it for a Queen.
- **Checkmate / stalemate** - "no legal move and in check" / "no legal move and not in check".
- **Draw by repetition / 50-move rule** - needs a position hash, which is what chess engines do.

Notice how each of these slots cleanly into our existing classes. That's the win: the
design we built in Notebook 1 can *grow* without a rewrite.


## OOD takeaways

1. **Polymorphism beats `if/elif` chains.** Each piece knows its own rules.
2. **Separate concerns.** `Board` stores, `Piece` decides moves, `Game` enforces order.
3. **Model actions as objects.** `Move` makes undo and history trivial.
4. **Reuse knowledge.** `in_check` reuses `valid_moves` - no duplicate logic.
5. **Separate geometry from legality.** A piece answers "where could I go?";
   the `Game` answers "may you?". Keeping those two questions in different
   classes is what makes pins and self-check fall out for free instead of
   needing special cases inside every piece.
6. **Leave room to grow.** Castling, en-passant, promotion all fit without breaking the shape.

If you remember only one thing: **put the knowledge where it naturally lives.**
